<a href="https://colab.research.google.com/github/grmntfrancis0/earthengine-community/blob/main/Juba_Floods_ee_auth_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --upgrade --pre xee

In [ ]:
!pip install -U geemap

In [ ]:
import ee

In [ ]:
ee.Authenticate()
ee.Initialize(project = "ee-grmntfrancis0",
              opt_url = 'https://earthengine-highvolume.googleapis.com')

In [ ]:
import geemap

In [ ]:
map = geemap.Map(basemap = "SATELLITE")
map

In [ ]:
roi = map.draw_last_feature.geometry()
roi

In [ ]:
def ndwi(img):
  qa_band = img.select('QA_PIXEL')
  bit1 = qa_band.bitwiseAnd(1 << 1).neq(0)
  bit2 = qa_band.bitwiseAnd(1 << 2).neq(0)
  bit3 = qa_band.bitwiseAnd(1 << 3).neq(0)
  bit4 = qa_band.bitwiseAnd(1 << 4).neq(0)
  mask = bit1.Or(bit2).Or(bit3).Or(bit4)
  sr = img.select('SR.*').multiply(2.75e-05).add(-0.2)
  index = sr.normalizedDifference(['SR_B3','SR_B5']).rename('ndwi')
  return index.updateMask(mask.Not()).copyProperties(img,['system:time_start'])

In [ ]:
landsat = (
    ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
    .filterDate('2018','2020')
    .filterBounds(roi)
)

In [ ]:
landsat_ndwi = landsat.map(ndwi)

In [ ]:
landsat_ndwi

In [ ]:
from shapely.geometry import shape

roi_shape = shape(roi.getInfo())
roi_shape

In [ ]:
from xee import helpers

landsat.first().select('SR_B4').projection()

grid_params = helpers.fit_geometry(
    geometry = roi_shape,
    grid_crs = 'EPSG:32639',
    grid_scale = (100, -100)
)

In [ ]:
import xarray as xr

In [ ]:
ds = xr.open_dataset(
    landsat_ndwi,
    engine ='ee',
    **grid_params
)

In [ ]:
ds = ds.sortby('time') * 1

In [ ]:
baseline = ds.median(dim = 'time')

In [ ]:
baseline.ndwi.plot(x = 'x', y ='y', robust = True)

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
baseline.ndwi.values.flatten()

In [ ]:
plt.hist(baseline.ndwi.values.flatten(), bins = 200)
plt.show()

flood = ds.sel(time = '2018-04').max(dim = 'time')


In [ ]:
flood.ndwi.plot(x = 'x', y = 'y', robust = True)

In [ ]:
plt.hist(flood.ndwi.values.flatten(), bins = 200)
plt.show()


In [ ]:
fig, ax = plt.subplots(1, 3, figsize = (13,4))

baseline.ndwi.plot(x = 'x', y = 'y', robust = True, ax = ax[0],
                   cbar_kwargs = {'orientation':'horizontal', 'pad': 0.05,
                                  'label':'pre-flood'})


ax[0].tick_params(left = False, bottom = False, labelleft = False, labelbottom = False)
ax[0].set_ylabel('')
ax[0].set_xlabel('')

flood.ndwi.plot(x = 'x', y ='y', robust = True, ax = ax[2],
                cbar_kwargs = {'orientation': 'horizontal', 'pad': 0.05,
                               'label':'post-flood'})


ax[2].tick_params(left = False, labelleft = False, bottom = False, labelbottom = False)
ax[2].set_ylabel('')
ax[2].set_xlabel('')

ax[1].hist(flood.ndwi.values.flatten(), bins = 200, color = 'tab:blue', label = 'post-flood', alpha = 0.5)
ax[1].hist(baseline.ndwi.values.flatten(), bins = 200, color = 'tab:red', label = 'pre-flood', alpha = 0.5)
ax[1].legend()

plt.savefig('flood_hist.png', dpi = 360, bbox_inches ='tight')
plt.show()